# MA3001B • Semana 04: Redes Secuenciales y Transformers
## RNN, GRU, LSTM y Mecanismo de Auto-Atención en Series de Tiempo
**Profesor B: Modelos y TDA • Tecnológico de Monterrey**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alxcn/Models_MA3001B/blob/main/notebooks/N1_RedesSecuenciales_Transformers.ipynb)

---
### Objetivos de la Sesión
1. Explicar el fenómeno de desvanecimiento del gradiente en RNN simples con ventanas largas.
2. Comprender la mitigación mediante compuertas en LSTM ($f, i, o$) y GRU ($z, r$).
3. Implementar el mecanismo de auto-atención multi-cabeza y extraer la matriz de atención para inspeccionar rezagos clave.
4. Entrenar y comparar RNN, GRU, LSTM y Transformer frente a un **baseline ingenuo** usando corte cronológico estricto sin fuga de información.


## 1. Problema Detonador: Pérdida Estancada y Aplanamiento
Entrenamos un modelo recurrente simple (RNN) con ventanas largas ($L=96$) sobre una serie con tendencia. Observemos cómo la pérdida se estanca y el pronóstico degenera en una línea casi recta, mientras que un baseline ingenuo que simplemente repite el valor anterior logra un error drásticamente menor.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# Cargamos serie de prueba
y_det = sm.datasets.co2.load_pandas().data["co2"].resample("MS").mean().interpolate()

# Ventana muy larga L = 96
L_largo = 96
scaler_det = StandardScaler().fit(y_det.iloc[:int(len(y_det)*0.8)].to_frame())
z_det = scaler_det.transform(y_det.to_frame()).ravel().astype("float32")

def make_windows_np(z, L):
    X = np.stack([z[i:i + L] for i in range(len(z) - L)])
    t = z[L:]
    return torch.tensor(X, dtype=torch.float32).unsqueeze(-1), torch.tensor(t, dtype=torch.float32).unsqueeze(-1)

X_det, t_det = make_windows_np(z_det, L_largo)
n_tr = int(len(y_det) * 0.8) - L_largo

# RNN simple
rnn_bad = nn.RNN(input_size=1, hidden_size=16, batch_first=True)
head_bad = nn.Linear(16, 1)
opt_bad = torch.optim.Adam(list(rnn_bad.parameters()) + list(head_bad.parameters()), lr=1e-3)

dl_bad = DataLoader(TensorDataset(X_det[:n_tr], t_det[:n_tr]), batch_size=32, shuffle=True)
for epoch in range(15):
    for xb, yb in dl_bad:
        opt_bad.zero_grad()
        out, _ = rnn_bad(xb)
        loss = nn.MSELoss()(head_bad(out[:, -1, :]), yb)
        loss.backward()
        opt_bad.step()

with torch.no_grad():
    out_test, _ = rnn_bad(X_det[n_tr:])
    pred_bad = head_bad(out_test[:, -1, :]).squeeze(-1).numpy()
    true_test = t_det[n_tr:].squeeze(-1).numpy()
    naive_pred = X_det[n_tr:, -1, 0].numpy()

rmse_bad = np.sqrt(np.mean((true_test - pred_bad)**2))
rmse_naive = np.sqrt(np.mean((true_test - naive_pred)**2))

print(f"RMSE RNN simple (L=96): {rmse_bad:.4f}")
print(f"RMSE Baseline Ingenuo: {rmse_naive:.4f}")
print("-> ¿Por qué la RNN simple no logra superar al valor anterior y sufre con ventanas largas?")


## 2. Marco Teórico Mínimo

### 2.1 Ecuaciones de RNN, LSTM y GRU

**RNN Simple:**
$$h_t = \tanh(W_x x_t + W_h h_{t-1} + b), \qquad \hat{y}_t = W_y h_T + c$$

**LSTM:**
$$\begin{aligned}
f_t &= \sigma(W_f [h_{t-1}, x_t] + b_f) \quad &\text{(Olvido)} \\
i_t &= \sigma(W_i [h_{t-1}, x_t] + b_i) \quad &\text{(Entrada)} \\
o_t &= \sigma(W_o [h_{t-1}, x_t] + b_o) \quad &\text{(Salida)} \\
\tilde{c}_t &= \tanh(W_c [h_{t-1}, x_t] + b_c), \quad &c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \\
h_t &= o_t \odot \tanh(c_t)
\end{aligned}$$

**GRU:**
$$\begin{aligned}
z_t &= \sigma(W_z [h_{t-1}, x_t]), \quad r_t = \sigma(W_r [h_{t-1}, x_t]) \\
\tilde{h}_t &= \tanh(W [r_t \odot h_{t-1}, x_t]), \quad h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t
\end{aligned}$$

### 2.2 Mecanismo de Auto-Atención
$$\operatorname{Attention}(Q, K, V) = \operatorname{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$


### Configuración de Datos: Práctica (CO₂) o Serie del Reto
En la siguiente celda se carga la serie de práctica por defecto (**CO₂ mensual de Mauna Loa** de `statsmodels`).
Para aplicar el análisis a la serie de su equipo en el Reto, cambie la bandera `USAR_SERIE_RETO = True` y proporcione la ruta a su archivo CSV.

In [ ]:
# ==============================================================================
# CELDA SERIE_RETO: Alternar entre datos de práctica y datos del Reto
# ==============================================================================
USAR_SERIE_RETO = False  # Cambiar a True para usar la serie de su Reto

if USAR_SERIE_RETO:
    # Sustituya con la carga de la serie de su equipo:
    # df_reto = pd.read_csv("datos_reto.csv", index_col=0, parse_dates=True)
    # y = df_reto["columna_objetivo"].asfreq("MS").interpolate()
    print("Modo Reto activado.")
else:
    # Serie de práctica por defecto: CO2 de Mauna Loa (mensual interpolado)
    y = sm.datasets.co2.load_pandas().data["co2"].resample("MS").mean().interpolate()
    print(f"Cargada serie de práctica CO2: {len(y)} observaciones ({y.index.min().strftime('%Y-%m')} a {y.index.max().strftime('%Y-%m')})")


## 3. Ejemplo Aplicado Paso a Paso

### Paso 1: Partición cronológica y escalamiento sin fuga de información

In [ ]:
# 1) División cronológica 80% entrenamiento / 20% prueba
n_train = int(len(y) * 0.8)

scaler = StandardScaler().fit(y.iloc[:n_train].to_frame())
z = scaler.transform(y.to_frame()).ravel().astype("float32")

L = 24
def make_windows(z, L):
    X = np.stack([z[i:i + L] for i in range(len(z) - L)])
    t = z[L:]
    return torch.tensor(X, dtype=torch.float32).unsqueeze(-1), torch.tensor(t, dtype=torch.float32).unsqueeze(-1)

X, t = make_windows(z, L)
split = n_train - L

train_ds = TensorDataset(X[:split], t[:split])
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)

print(f"Dimensiones de entrenamiento: X = {X[:split].shape}, t = {t[:split].shape}")
print(f"Dimensiones de prueba: X = {X[split:].shape}, t = {t[split:].shape}")


### Paso 2: Modelos Recurrentes (RNN, GRU, LSTM)

In [ ]:
class SeqModel(nn.Module):
    def __init__(self, kind="lstm", hidden=32):
        super().__init__()
        cells_dict = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}
        self.rnn = cells_dict[kind](input_size=1, hidden_size=hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1, :])

def fit_model(model, dl, epochs=35, lr=1e-3, clip=1.0):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    losses = []
    for ep in range(epochs):
        model.train()
        ep_loss = 0.0
        for xb, yb in dl:
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
            ep_loss += loss.item() * len(xb)
        losses.append(ep_loss / len(dl.dataset))
    return model, losses

def predict_model(model, X_eval):
    model.eval()
    with torch.no_grad():
        return model(X_eval).squeeze(-1).numpy()


### Paso 3: Transformer y TinyAttn para Inspeccionar Pesos

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.size(1)]

class TSTransformer(nn.Module):
    def __init__(self, d_model=32, nhead=4, num_layers=2, dim_ff=64, dropout=0.1):
        super().__init__()
        self.inp = nn.Linear(1, d_model)
        self.pos = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True
        )
        self.enc = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        h = self.enc(self.pos(self.inp(x)))
        return self.head(h[:, -1, :])

class TinyAttn(nn.Module):
    def __init__(self, d_model=32, nhead=4):
        super().__init__()
        self.inp = nn.Linear(1, d_model)
        self.pos = PositionalEncoding(d_model)
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        h = self.pos(self.inp(x))
        a, w = self.attn(h, h, h, need_weights=True)
        self.w = w.detach()
        return self.head(self.norm(h + a)[:, -1, :])


### Paso 4: Entrenamiento y Tabla de Resultados

In [ ]:
rmse_fn = lambda a, b: float(np.sqrt(np.mean((a - b) ** 2)))
mae_fn = lambda a, b: float(np.mean(np.abs(a - b)))

true_test = t[split:].squeeze(-1).numpy()
naive_pred = X[split:, -1, 0].numpy()

resultados = {
    "Ingenuo (y[t-1])": {"RMSE": rmse_fn(true_test, naive_pred), "MAE": mae_fn(true_test, naive_pred)}
}

for kind in ["rnn", "gru", "lstm"]:
    m, _ = fit_model(SeqModel(kind=kind), train_dl, epochs=35)
    preds = predict_model(m, X[split:])
    resultados[kind.upper()] = {"RMSE": rmse_fn(true_test, preds), "MAE": mae_fn(true_test, preds)}

m_tf, _ = fit_model(TSTransformer(), train_dl, epochs=35)
preds_tf = predict_model(m_tf, X[split:])
resultados["TRANSFORMER"] = {"RMSE": rmse_fn(true_test, preds_tf), "MAE": mae_fn(true_test, preds_tf)}

df_res = pd.DataFrame(resultados).T
print("=== RESULTADOS EN CONJUNTO DE PRUEBA (Escala Estandarizada) ===")
print(df_res.round(4))


### Paso 5: Gráfica del Perfil de Atención vs. Rezagos

In [ ]:
m_tiny, _ = fit_model(TinyAttn(), train_dl, epochs=35)
_ = predict_model(m_tiny, X[split:])

attn_profile = m_tiny.w[:, -1, :].mean(dim=0).cpu().numpy()
rezagos = np.arange(L, 0, -1)

plt.figure(figsize=(9, 4))
plt.bar(rezagos, attn_profile, color="#CB4B16", alpha=0.85, edgecolor="#073642")
plt.xlabel("Rezago hacia el pasado (k pasos)")
plt.ylabel("Peso de Atención Promedio")
plt.title("Perfil de Atención del Último Paso (TinyAttn) vs. Rezagos")
plt.grid(True, linestyle="--", alpha=0.5)
plt.gca().invert_xaxis()
plt.tight_layout()
plt.show()

print(f"Rezagos más atendidos: {rezagos[np.argsort(attn_profile)[::-1][:4]]}")


## 4. Actividad del Reto y Entregable

### Instrucciones:
1. Pruebe variar la ventana a $L=12$ y $L=48$.
2. Evalúe qué sucede si entrena la LSTM sobre los datos de su Reto.
3. Complete las celdas `# TODO`.

In [ ]:
# ==============================================================================
# TODO 1: Comparar L=12 vs L=48 en su serie
# ==============================================================================
# Ajuste dos modelos LSTM (uno con L=12 y otro con L=48).
# Imprima sus RMSE en el conjunto de prueba.


In [ ]:
# ==============================================================================
# TODO 2: Análisis crítico del baseline ingenuo
# ==============================================================================
# Explique por qué el baseline ingenuo supera a varios modelos neuronales en esta serie:
# [Escriba su explicación aquí]


## 5. Conclusiones y Pregunta Abierta
- **Conclusión:** Las redes neuronales profundas en series con tendencia requieren técnicas rigurosas para no quedar rezagadas frente a baselines estadísticos simples.
- **Pregunta para Semana 05:** ¿Cómo eliminamos formalmente la tendencia y estacionalidad mediante diferenciación y descomposición STL para estabilizar las predicciones?
